# Week 22, Day 5: Attractors, integrators and what trained networks learn

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w022/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Memory time from the eigenvalue

For w = 0.9 and 0.99, start a unit (tau = 20 ms) at r = 1, simulate 2 s with Euler steps of 1 ms and no input, and measure `tau_eff[w]` from the slope of log r.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
tau = 20.0
tau_eff = {}
for w in (0.9, 0.99):
    ts = np.arange(0, 2000, 1.0); r = np.empty(len(ts)); r[0] = 1.0
    for k in range(1, len(ts)):
        r[k] = ...
    tau_eff[w] = ...
    plt.semilogy(ts, r, label=f"w = {w}")
print({w: round(float(t), 1) for w, t in tau_eff.items()}, " ms measured;", {w: round(tau / (1 - w), 1) for w in tau_eff}, " ms predicted")
plt.xlabel("ms"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_match():
    "the measured time constants are within 2% of tau / (1 - w)"
    for w in (0.9, 0.99):
        assert abs(tau_eff[w] - 20 / (1 - w)) / (20 / (1 - w)) < 0.02

test_match()
print("All checks passed")

### Exercise: Integrating on a line

Simulate 1 s with tau = 20 ms and dt = 0.1 ms, giving input (2, 0) from 100 to 150 ms, (0, 1) from 400 to 420 ms and (−1, −1) from 700 to 710 ms. Store the trajectory in `R`. The final mean of r1 and r2 should equal `target`, the time integral of the mean input divided by tau: compute it.

In [ ]:
W2 = np.array([[0.5, 0.5], [0.5, 0.5]])
dt = 0.1; ts = np.arange(0, 1000, dt)
def inp(t):
    if 100 <= t < 150: return np.array([2.0, 0.0])
    if 400 <= t < 420: return np.array([0.0, 1.0])
    if 700 <= t < 710: return np.array([-1.0, -1.0])
    return np.zeros(2)
R = np.zeros((len(ts), 2))
for k in range(1, len(ts)):
    R[k] = ...
target = ...                                      # (50 * 1 + 20 * 0.5 + 10 * (-1)) / tau
print(f"final state {R[-1].round(4)}, predicted position on the line {target:.4f}")
plt.plot(R[:, 0], R[:, 1]); plt.plot([-1, 3], [-1, 3], "k:"); plt.xlabel("r1"); plt.ylabel("r2"); plt.axis("equal"); plt.show()

In [ ]:
import numpy as np
def test_on_line():
    "after the inputs end the state is on the line r1 = r2 to within 1e-3"
    assert abs(R[-1, 0] - R[-1, 1]) < 1e-3

def test_sum():
    "the position along the line equals the integral of the input along (1, 1) / 2 to within 1%"
    assert abs(R[-1].mean() - target) / abs(target) < 0.01

test_on_line()
test_sum()
print("All checks passed")

### Exercise: Diffusion along, not across

Simulate 2,000 independent trials of the two-neuron line attractor at once for 1 s (dt = 0.1 ms) with independent noise of size sigma = 0.05 added to each neuron as sigma √dt times a normal draw. Every 1 ms, store the variance across trials of the component along (1, 1)/√2 in `var_along` and along (1, −1)/√2 in `var_across`.

In [ ]:
r_n = np.random.default_rng(1996)
n_tr, sigma = 2000, 0.05
X = np.zeros((n_tr, 2)); var_along, var_across = [], []
u_along, u_across = np.array([1, 1]) / np.sqrt(2), np.array([1, -1]) / np.sqrt(2)
for k in range(10000):
    X = X + dt / tau * (-X + X @ W2.T) + ...
    if k % 10 == 9:
        var_along.append(np.var(X @ u_along)); var_across.append(...)
var_along, var_across = np.array(var_along), np.array(var_across)
print(f"along the line: variance {var_along[499]:.4f} at 500 ms, {var_along[-1]:.4f} at 1 s;  across: {var_across[499]:.5f} and {var_across[-1]:.5f}")
plt.plot(var_along, label="along"); plt.plot(var_across, label="across"); plt.xlabel("ms"); plt.ylabel("variance"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_along():
    "variance along the line roughly doubles from 500 to 1000 ms (ratio between 1.6 and 2.4)"
    assert 1.6 < var_along[-1] / var_along[len(var_along) // 2] < 2.4

def test_across():
    "variance across the line has saturated (ratio between 0.8 and 1.25)"
    assert 0.8 < var_across[-1] / var_across[len(var_across) // 2] < 1.25

test_along()
test_across()
print("All checks passed")

### Exercise: Find the hidden line

Write `grad_q(x)` = Jᵀ F(x), where J = −I + W diag(1 − tanh²(x)) is the Jacobian of F. Run 2,000 gradient steps of size 0.2 from each of 40 random starts, keep the end points with q below 1e-5 in `found`, and store the first right singular vector of `found` in `pc1`.

In [ ]:
r_s = np.random.default_rng(2013); N = 20
u_hidden = r_s.normal(size=N); u_hidden /= np.linalg.norm(u_hidden)
P = np.eye(N) - np.outer(u_hidden, u_hidden)
W = np.outer(u_hidden, u_hidden) + 0.5 * P @ (r_s.normal(size=(N, N)) / np.sqrt(N)) @ P
Fn = lambda x: -x + W @ np.tanh(x)
q = lambda x: 0.5 * np.sum(Fn(x) ** 2)

def grad_q(x):
    J = ...
    return J.T @ Fn(x)

ends = []
for _ in range(40):
    x = r_s.normal(size=N)
    for _ in range(2000):
        x = x - 0.2 * grad_q(x)
    ends.append(x)
found = np.array([x for x in ends if q(x) < 1e-5])
pc1 = ...
print(f"{len(found)} slow points;  |cos(pc1, u)| = {abs(pc1 @ u_hidden):.4f};  positions along u: {np.sort(found @ u_hidden).round(2)}")

In [ ]:
import numpy as np
def test_found():
    "at least 10 of the 40 searches end at points with q below 1e-5"
    assert len(found) >= 10

def test_line():
    "the found points lie on a line: the first singular vector explains over 99% and aligns with u (|cos| above 0.99)"
    S = np.linalg.svd(found, compute_uv=False)
    assert S[0] ** 2 / np.sum(S ** 2) > 0.99 and abs(pc1 @ u_hidden) > 0.99

test_found()
test_line()
print("All checks passed")